In [28]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.linear_model import LogisticRegression
import pickle
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures,
)

from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from geoai.utils_ds.PreProcessingOps import PreProcessingOperations
from geoai.utils_ml.ModelOps import ModelOperations

preprocess_ops = PreProcessingOperations()
model_ops = ModelOperations()


In [29]:
X_train = pd.read_csv("csv_files/X_train.csv")
X_test = pd.read_csv("csv_files/X_test.csv")
y_train = pd.read_csv("csv_files/y_train.csv")
y_test = pd.read_csv("csv_files/y_test.csv")

- Compute indices
- Bin NDVI
- Categorize NDVI
- One-hot encoding on Bin NDVI
- Ordinal encoding on Categorize NDVI
- Polynomial transformation
- normalize
- LDA
- Fit and predict

In [30]:
X_train["NDVI"] = (X_train["NIR"] - X_train["RED"]) / (X_train["NIR"] + X_train["RED"])
X_train["NDBI"] = (X_train["SWIR"] - X_train["NIR"]) / (X_train["SWIR"] + X_train["NIR"])
X_train["REI"] = (X_train["NIR"] - X_train["BLUE"]) / (X_train["NIR"] + X_train["BLUE"] * X_train["NIR"])
X_train.fillna(0, inplace=True)

X_test["NDVI"] = (X_test["NIR"] - X_test["RED"]) / (X_test["NIR"] + X_test["RED"])
X_test["NDBI"] = (X_test["SWIR"] - X_test["NIR"]) / (X_test["SWIR"] + X_test["NIR"])
X_test["REI"] = (X_test["NIR"] - X_test["BLUE"]) / (X_test["NIR"] + X_test["BLUE"] * X_test["NIR"])
X_test.fillna(0, inplace=True)

In [31]:
# create binary NDVI
ndvi_binary_edges = [-float("inf"), 0.5, float("inf")]
ndvi_binary_labels = ["non_veg", "veg"]
X_train, X_test = preprocess_ops.binarize_or_discretize(
    X_train, X_test, "NDVI", "NDVI_bin", ndvi_binary_edges, ndvi_binary_labels
)

# create categorical NDVI
ndvi_category_edges = [-float("inf"), 0.2, 0.5, float("inf")]
ndvi_category_labels = ["low_veg", "medium_veg", "high_veg"]
X_train,X_test = preprocess_ops.binarize_or_discretize(
    X_train, X_test, "NDVI", "NDVI_dis", ndvi_category_edges, ndvi_category_labels
)

In [32]:
X_train.head(1)

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,NDVI_bin,NDVI_dis
0,373.5,519.5,389.0,3034.0,1991.0,0.772714,-0.207562,0.002342,veg,high_veg


In [33]:
create_bins_columns = ["NDVI"]
numerical_columns = ["BLUE", "GREEN", "RED", "NIR", "SWIR", "NDVI", "NDBI", "REI"]
one_hot_encoder_columns = ["NDVI_bin"]
ordinal_encoder_columns = ["NDVI_dis"]
categories = [["low_veg", "medium_veg", "high_veg"]]
best_params = {"C": 9.232675920286502, "max_iter": 160, "solver": "lbfgs"}

numerical_transformer = Pipeline(
    steps=[("poly", PolynomialFeatures(degree=2))]
)

one_hot_transformer = OneHotEncoder(dtype=int)
ordinal_transformer = OrdinalEncoder(categories=categories, dtype=int)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_columns),
        ("onehot", one_hot_transformer, one_hot_encoder_columns),
        ("ordinal", ordinal_transformer, ordinal_encoder_columns),
    ],  remainder='drop'  
)


pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("scale", MinMaxScaler()),
        ("dim_reduce", LinearDiscriminantAnalysis(n_components=3)),
        ("classifier", LogisticRegression(**best_params)),
    ]
)

In [34]:
# Fit the pipeline to the training data
pipeline.fit(X_train, y_train)

# Transform the test data and make predictions
y_train_pred = pipeline.predict(X_train)
y_pred_test = pipeline.predict(X_test)

# calculate the accuracy
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_train, y_train_pred)}")
print(f"Test Accuracy: {model_ops.calculate_classification_accuracy(y_test, y_pred_test)}")

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Train Accuracy: (0.937467700258398, 0.9378469707503807, 0.937467700258398, 0.9375970119680029)
Test Accuracy: (0.9256198347107438, 0.9265542124090291, 0.9256198347107438, 0.9258873031024559)


d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [35]:
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('poly',
                                                                   PolynomialFeatures())]),
                                                  ['BLUE', 'GREEN', 'RED',
                                                   'NIR', 'SWIR', 'NDVI',
                                                   'NDBI', 'REI']),
                                                 ('onehot',
                                                  OneHotEncoder(dtype=<class 'int'>),
                                                  ['NDVI_bin']),
                                                 ('ordinal',
                                                  OrdinalEncoder(categories=[['low_veg',
                                                                              'medium_veg',
                                                                              'high_veg']],
                                                                 dtype=<class 'int'>),
                                                  ['NDVI_dis'])])),
                ('scale', MinMaxScaler()),
                ('dim_reduce', LinearDiscriminantAnalysis(n_components=3)),
                ('classifier',
                 LogisticRegression(C=9.232675920286502, max_iter=160))])

In [36]:
# save the model using pickle
# merge the train and test datasets

X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])

# train the model using the best hyperparameters and the whole dataset
pipeline.fit(X_all, y_all)
with open('final_pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
